In [1]:
# # ============================================================================
# # CELL 1: Install Dependencies
# # ============================================================================
# !pip install -q segmentation-models-pytorch

# # ============================================================================
# # CELL 2: Imports and Setup
# # ============================================================================
# import json
# import random
# import time
# from pathlib import Path

# import cv2
# import numpy as np
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader

# import segmentation_models_pytorch as smp

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device: {DEVICE}")

# def seed_everything(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# # ============================================================================
# # CELL 3: Configuration
# # ============================================================================
# ARCHITECTURES = {
#     "unet": smp.Unet,
#     "deeplabv3plus": smp.DeepLabV3Plus,
# }

# ENCODERS = {
#     "efficientnet_b0": "efficientnet-b0",
#     "efficientnet_b1": "efficientnet-b1",
#     "efficientnet_b2": "efficientnet-b2",
#     "efficientnet_b3": "efficientnet-b3",
#     "efficientnet_b4": "efficientnet-b4",
#     # "efficientnet_b5": "efficientnet-b5",
#     # "efficientnet_b6": "efficientnet-b6",
#     # "efficientnet_b7": "efficientnet-b7",
#     # "mobilenet_v2": "mobilenet_v2",
#     # "resnet_18": "resnet18",
#     # "resnet_34": "resnet34",
#     # "resnet_50": "resnet50",
# }

# SEEDS = [42, 1337, 2025]
# # SEEDS = [42, 1337]

# CONFIG = {
#     "img_size": 224,
#     "batch_size": 16,
#     "epochs": 50,
#     "lr": 1e-4,
#     "weight_decay": 1e-4,
#     "patience": 10,
#     "min_lr": 1e-6,
#     "num_workers": 4,
# }

# # ============================================================================
# # CELL 4: Data Paths
# # ============================================================================
# DATA_ROOT = Path("/kaggle/input/dataset-skripsi-restart/data/processed/224x224")
# IMG_DIR = DATA_ROOT / "images"
# MSK_DIR = DATA_ROOT / "masks"
# SPLIT_FILE = DATA_ROOT.parent.parent.parent / "splits.json"

# # ============================================================================
# # CELL 5: Dataset (Normalization Only)
# # ============================================================================
# class LungDataset(Dataset):
#     def __init__(self, ids, img_dir, msk_dir):
#         self.ids = ids
#         self.img_dir = img_dir
#         self.msk_dir = msk_dir
#         # ImageNet normalization
#         self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
#         self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

#     def __len__(self):
#         return len(self.ids)

#     def __getitem__(self, idx):
#         img_id = self.ids[idx]
#         img = cv2.imread(str(self.img_dir / img_id), cv2.IMREAD_GRAYSCALE)
#         img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
#         mask = cv2.imread(str(self.msk_dir / img_id), cv2.IMREAD_GRAYSCALE)
#         mask = (mask > 0).astype(np.uint8)
        
#         # Convert to tensor and normalize
#         img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
#         img = (img - self.mean) / self.std
        
#         mask = torch.from_numpy(mask).float().unsqueeze(0)
        
#         return img, mask

# # ============================================================================
# # CELL 6: Load Data
# # ============================================================================
# with open(SPLIT_FILE, "r") as f:
#     splits = json.load(f)

# train_ids, val_ids = splits["train"], splits["val"]

# train_loader = DataLoader(
#     LungDataset(train_ids, IMG_DIR, MSK_DIR),
#     batch_size=CONFIG["batch_size"], shuffle=True, 
#     num_workers=CONFIG["num_workers"], pin_memory=True
# )

# val_loader = DataLoader(
#     LungDataset(val_ids, IMG_DIR, MSK_DIR),
#     batch_size=CONFIG["batch_size"], shuffle=False,
#     num_workers=CONFIG["num_workers"], pin_memory=True
# )

# print(f"Train: {len(train_ids)}, Val: {len(val_ids)}")

# # ============================================================================
# # CELL 7: Model Components
# # ============================================================================
# def build_model(architecture_class, encoder):
#     return architecture_class(
#         encoder_name=encoder, encoder_weights="imagenet",
#         in_channels=3, classes=1, activation=None
#     ).to(DEVICE)

# dice_loss = smp.losses.DiceLoss(mode="binary")
# bce_loss = nn.BCEWithLogitsLoss()

# def criterion(logits, targets):
#     return 0.5 * dice_loss(logits, targets) + 0.5 * bce_loss(logits, targets)

# # ============================================================================
# # CELL 8: Training and Validation
# # ============================================================================
# def train_one_epoch(model, loader, optimizer):
#     model.train()
#     total_loss = 0.0
#     for x, y in loader:
#         x, y = x.to(DEVICE), y.to(DEVICE)
#         optimizer.zero_grad()
#         loss = criterion(model(x), y)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     return total_loss / len(loader)

# @torch.no_grad()
# def validate(model, loader):
#     model.eval()
#     total_loss = 0.0
#     for x, y in loader:
#         x, y = x.to(DEVICE), y.to(DEVICE)
#         logits = model(x)
#         loss = criterion(logits, y)
#         total_loss += loss.item()
#     return total_loss / len(loader)

# # ============================================================================
# # CELL 9: Early Stopping
# # ============================================================================
# class EarlyStopping:
#     def __init__(self, patience=15):
#         self.patience = patience
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
        
#     def __call__(self, score):
#         if self.best_score is None:
#             self.best_score = score
#             return False
        
#         if score < self.best_score:  # Lower loss is better
#             self.best_score = score
#             self.counter = 0
#             return False
#         else:
#             self.counter += 1
#             if self.counter >= self.patience:
#                 self.early_stop = True
#             return self.early_stop

# # ============================================================================
# # CELL 10: Main Training Loop
# # ============================================================================
# EXP_ROOT = Path("experiments")
# EXP_ROOT.mkdir(exist_ok=True)

# for arch_name, arch_class in ARCHITECTURES.items():
#     for enc_name, enc_id in ENCODERS.items():
#         for seed in SEEDS:
#             print(f"\n{'='*60}\n{arch_name} | {enc_name} | Seed {seed}\n{'='*60}")
            
#             seed_everything(seed)
#             run_dir = EXP_ROOT / arch_name / enc_name / f"seed{seed}"
#             run_dir.mkdir(parents=True, exist_ok=True)
            
#             # Save config
#             with open(run_dir / "config.json", "w") as f:
#                 json.dump({**CONFIG, "architecture": arch_name, "encoder": enc_id, "seed": seed}, f, indent=2)
            
#             # Build model
#             model = build_model(arch_class, enc_id)
#             optimizer = torch.optim.AdamW(
#                 model.parameters(), 
#                 lr=CONFIG["lr"], 
#                 weight_decay=CONFIG["weight_decay"]
#             )
#             scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#                 optimizer, mode='min', factor=0.5, patience=5, min_lr=CONFIG["min_lr"]
#             )
#             early_stopping = EarlyStopping(patience=CONFIG["patience"])
            
#             # Training
#             history = []
#             best_val_loss = float('inf')
#             train_start = time.time()
            
#             for epoch in range(CONFIG["epochs"]):
#                 epoch_start = time.time()
                
#                 train_loss = train_one_epoch(model, train_loader, optimizer)
#                 val_loss = validate(model, val_loader)
#                 scheduler.step(val_loss)
                
#                 epoch_data = {
#                     "epoch": epoch + 1,
#                     "train_loss": train_loss,
#                     "val_loss": val_loss,
#                     "learning_rate": optimizer.param_groups[0]['lr'],
#                     "epoch_time_sec": time.time() - epoch_start,
#                 }
#                 history.append(epoch_data)
                
#                 # Save best model
#                 if val_loss < best_val_loss:
#                     best_val_loss = val_loss
#                     torch.save(model.state_dict(), run_dir / "model_best.pth")
                
#                 print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
#                       f"Val Loss: {val_loss:.4f} | "
#                       f"LR: {optimizer.param_groups[0]['lr']:.2e}")
                
#                 # Early stopping
#                 if early_stopping(val_loss):
#                     print(f"Early stopping at epoch {epoch+1}")
#                     break
            
#             total_train_time = time.time() - train_start
            
#             # Add training summary to history
#             history.append({
#                 "summary": {
#                     "best_val_loss": best_val_loss,
#                     "total_time_sec": total_train_time,
#                     "total_time_min": total_train_time / 60,
#                     "epochs_trained": len(history) - 1,  # -1 because we're adding summary
#                 }
#             })
            
#             # Save history
#             with open(run_dir / "history.json", "w") as f:
#                 json.dump(history, f, indent=2)
            
#             print(f"Best Val Loss: {best_val_loss:.4f} | "
#                   f"Time: {total_train_time/60:.1f}min")

# print(f"\n✅ Complete! Results in: {EXP_ROOT}")

In [2]:
# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
!pip install -q segmentation-models-pytorch
!pip install -q timm  # Add this line

# ============================================================================
# CELL 2: Imports and Setup
# ============================================================================
import json
import random
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# CELL 3: Configuration
# ============================================================================
ARCHITECTURES = {
    "unet": smp.Unet,
    "deeplabv3plus": smp.DeepLabV3Plus,
}

ENCODERS = {
    # "efficientnet_b0": "efficientnet-b0",
    # "efficientnet_b1": "efficientnet-b1",
    # "efficientnet_b2": "efficientnet-b2",
    # "efficientnet_b3": "efficientnet-b3",
    # "efficientnet_b4": "efficientnet-b4",
    # "efficientnet_b5": "efficientnet-b5",
    # "efficientnet_b6": "efficientnet-b6",
    # "efficientnet_b7": "efficientnet-b7",
    # "mobilenet_v2": "mobilenet_v2",
    # "resnet_18": "resnet18",
    # "resnet_34": "resnet34",
    # "resnet_50": "resnet50",

    "mobilenetv3_small_minimal": "timm-mobilenetv3_small_minimal_100",
    "mobilenetv3_small": "timm-mobilenetv3_small_100",
    "mobilenetv3_large_minimal": "timm-mobilenetv3_large_minimal_100",
    "mobilenetv3_large": "timm-mobilenetv3_large_100",
}

SEEDS = [42, 
         1337, 
         2025
        ]

CONFIG = {
    "img_size": 224,
    "batch_size": 16,
    "epochs": 50,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "patience": 10,
    "min_lr": 1e-6,
    "num_workers": 4,
}

# Class definitions
NUM_CLASSES = 4  # Background, Left Lung, Right Lung, Heart
CLASS_VALUES = {
    0: 0,     # Background
    85: 1,    # Left Lung
    170: 2,   # Right Lung
    255: 3,   # Heart
}

# ============================================================================
# CELL 4: Data Paths
# ============================================================================
DATA_ROOT = Path("/kaggle/input/dataset-skripsi-jsrt/data/processed/224x224")
IMG_DIR = DATA_ROOT / "images"
MSK_DIR = DATA_ROOT / "masks"
SPLIT_FILE = DATA_ROOT.parent.parent.parent / "splits.json"

# ============================================================================
# CELL 5: Dataset with Multi-Class Support
# ============================================================================
class LungDataset(Dataset):
    def __init__(self, ids, img_dir, msk_dir):
        self.ids = ids
        self.img_dir = img_dir
        self.msk_dir = msk_dir
        # ImageNet normalization
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.ids)
    
    def grayscale_to_class(self, mask):
        """Convert grayscale mask (0, 85, 170, 255) to class indices (0, 1, 2, 3)"""
        class_mask = np.zeros_like(mask, dtype=np.int64)
        
        for gray_value, class_idx in CLASS_VALUES.items():
            class_mask[mask == gray_value] = class_idx
        
        return class_mask

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        
        # Load image
        img = cv2.imread(str(self.img_dir / img_id), cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        # Load mask and convert to class indices
        mask = cv2.imread(str(self.msk_dir / img_id), cv2.IMREAD_GRAYSCALE)
        mask = self.grayscale_to_class(mask)
        
        # Convert to tensor and normalize
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        img = (img - self.mean) / self.std
        
        mask = torch.from_numpy(mask).long()  # Shape: (H, W)
        
        return img, mask

# ============================================================================
# CELL 6: Load Data
# ============================================================================
with open(SPLIT_FILE, "r") as f:
    splits = json.load(f)

train_ids = splits["root"]["train"]
val_ids = splits["root"]["val"]

train_loader = DataLoader(
    LungDataset(train_ids, IMG_DIR, MSK_DIR),
    batch_size=CONFIG["batch_size"], shuffle=True, 
    num_workers=CONFIG["num_workers"], pin_memory=True
)

val_loader = DataLoader(
    LungDataset(val_ids, IMG_DIR, MSK_DIR),
    batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=True
)

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}")

# ============================================================================
# CELL 7: Model Components
# ============================================================================
def build_model(architecture_class, encoder):
    return architecture_class(
        encoder_name=encoder, 
        encoder_weights="imagenet",
        in_channels=3, 
        classes=NUM_CLASSES,  # Changed from 1 to NUM_CLASSES
        activation=None
    ).to(DEVICE)

# Multi-class losses
dice_loss = smp.losses.DiceLoss(mode="multiclass")
ce_loss = nn.CrossEntropyLoss()

def criterion(logits, targets):
    """Combined loss for multi-class segmentation"""
    # logits: (B, NUM_CLASSES, H, W)
    # targets: (B, H, W) with class indices
    return 0.5 * dice_loss(logits, targets) + 0.5 * ce_loss(logits, targets)

# ============================================================================
# CELL 8: Training and Validation
# ============================================================================
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()
    return total_loss / len(loader)

# ============================================================================
# CELL 9: Early Stopping
# ============================================================================
class EarlyStopping:
    def __init__(self, patience=15):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        
        if score < self.best_score:  # Lower loss is better
            self.best_score = score
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return self.early_stop

# ============================================================================
# CELL 10: Main Training Loop
# ============================================================================
EXP_ROOT = Path("experiments")
EXP_ROOT.mkdir(exist_ok=True)

for arch_name, arch_class in ARCHITECTURES.items():
    for enc_name, enc_id in ENCODERS.items():
        for seed in SEEDS:
            print(f"\n{'='*60}\n{arch_name} | {enc_name} | Seed {seed}\n{'='*60}")
            
            seed_everything(seed)
            run_dir = EXP_ROOT / arch_name / enc_name / f"seed{seed}"
            run_dir.mkdir(parents=True, exist_ok=True)
            
            # Save config
            with open(run_dir / "config.json", "w") as f:
                json.dump({
                    **CONFIG, 
                    "architecture": arch_name, 
                    "encoder": enc_id, 
                    "seed": seed,
                    "num_classes": NUM_CLASSES,
                    "class_mapping": CLASS_VALUES
                }, f, indent=2)
            
            # Build model
            model = build_model(arch_class, enc_id)
            optimizer = torch.optim.AdamW(
                model.parameters(), 
                lr=CONFIG["lr"], 
                weight_decay=CONFIG["weight_decay"]
            )
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=5, min_lr=CONFIG["min_lr"]
            )
            early_stopping = EarlyStopping(patience=CONFIG["patience"])
            
            # Training
            history = []
            best_val_loss = float('inf')
            train_start = time.time()
            
            for epoch in range(CONFIG["epochs"]):
                epoch_start = time.time()
                
                train_loss = train_one_epoch(model, train_loader, optimizer)
                val_loss = validate(model, val_loader)
                scheduler.step(val_loss)
                
                epoch_data = {
                    "epoch": epoch + 1,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "learning_rate": optimizer.param_groups[0]['lr'],
                    "epoch_time_sec": time.time() - epoch_start,
                }
                history.append(epoch_data)
                
                # Save best model
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), run_dir / "model_best.pth")
                
                print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
                      f"Val Loss: {val_loss:.4f} | "
                      f"LR: {optimizer.param_groups[0]['lr']:.2e}")
                
                # Early stopping
                if early_stopping(val_loss):
                    print(f"Early stopping at epoch {epoch+1}")
                    break
            
            total_train_time = time.time() - train_start
            
            # Add training summary to history
            history.append({
                "summary": {
                    "best_val_loss": best_val_loss,
                    "total_time_sec": total_train_time,
                    "total_time_min": total_train_time / 60,
                    "epochs_trained": len(history) - 1,
                }
            })
            
            # Save history
            with open(run_dir / "history.json", "w") as f:
                json.dump(history, f, indent=2)
            
            print(f"Best Val Loss: {best_val_loss:.4f} | "
                  f"Time: {total_train_time/60:.1f}min")

print(f"\n✅ Complete! Results in: {EXP_ROOT}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device: cuda
Train: 268, Val: 38

unet | mobilenetv3_small_minimal | Seed 42


model.safetensors:   0%|          | 0.00/8.25M [00:00<?, ?B/s]

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


...

Epoch   1 | Train Loss: 0.8910 | Val Loss: 0.9886 | LR: 1.00e-04
Epoch   2 | Train Loss: 0.5481 | Val Loss: 0.6153 | LR: 1.00e-04
Epoch   3 | Train Loss: 0.3400 | Val Loss: 0.3174 | LR: 1.00e-04
Epoch   4 | Train Loss: 0.2463 | Val Loss: 0.2108 | LR: 1.00e-04
Epoch   5 | Train Loss: 0.1990 | Val Loss: 0.1692 | LR: 1.00e-04
Epoch   6 | Train Loss: 0.1702 | Val Loss: 0.1500 | LR: 1.00e-04
Epoch   7 | Train Loss: 0.1494 | Val Loss: 0.1341 | LR: 1.00e-04
Epoch   8 | Train Loss: 0.1349 | Val Loss: 0.1223 | LR: 1.00e-04
Epoch   9 | Train Loss: 0.1215 | Val Loss: 0.1147 | LR: 1.00e-04
Epoch  10 | Train Loss: 0.1119 | Val Loss: 0.1067 | LR: 1.00e-04
Epoch  11 | Train Loss: 0.1041 | Val Loss: 0.0999 | LR: 1.00e-04
Epoch  12 | Train Loss: 0.0971 | Val Loss: 0.0956 | LR: 1.00e-04
Epoch  13 | Train Loss: 0.0926 | Val Loss: 0.0914 | LR: 1.00e-04
Epoch  14 | Train Loss: 0.0870 | Val Loss: 0.0874 | LR: 1.00e-04
Epoch  15 | Train Loss: 0.0820 | Val Loss: 0.0836 | LR: 1.00e-04
Epoch  16 | Train Loss: 0

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Epoch   1 | Train Loss: 0.8731 | Val Loss: 1.0144 | LR: 1.00e-04
Epoch   2 | Train Loss: 0.5196 | Val Loss: 0.6504 | LR: 1.00e-04
Epoch   3 | Train Loss: 0.3242 | Val Loss: 0.3284 | LR: 1.00e-04
Epoch   4 | Train Loss: 0.2355 | Val Loss: 0.2175 | LR: 1.00e-04
Epoch   5 | Train Loss: 0.1909 | Val Loss: 0.1752 | LR: 1.00e-04
Epoch   6 | Train Loss: 0.1613 | Val Loss: 0.1502 | LR: 1.00e-04
Epoch   7 | Train Loss: 0.1425 | Val Loss: 0.1324 | LR: 1.00e-04
Epoch   8 | Train Loss: 0.1289 | Val Loss: 0.1227 | LR: 1.00e-04
Epoch   9 | Train Loss: 0.1173 | Val Loss: 0.1121 | LR: 1.00e-04
Epoch  10 | Train Loss: 0.1088 | Val Loss: 0.1048 | LR: 1.00e-04
Epoch  11 | Train Loss: 0.1014 | Val Loss: 0.0988 | LR: 1.00e-04
Epoch  12 | Train Loss: 0.0944 | Val Loss: 0.0932 | LR: 1.00e-04
Epoch  13 | Train Loss: 0.0878 | Val Loss: 0.0923 | LR: 1.00e-04
Epoch  14 | Train Loss: 0.0833 | Val Loss: 0.0880 | LR: 1.00e-04
Epoch  15 | Train Loss: 0.0780 | Val Loss: 0.0829 | LR: 1.00e-04
Epoch  16 | Train Loss: 0

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Epoch   1 | Train Loss: 0.9170 | Val Loss: 1.0347 | LR: 1.00e-04
Epoch   2 | Train Loss: 0.5371 | Val Loss: 0.6485 | LR: 1.00e-04
Epoch   3 | Train Loss: 0.3336 | Val Loss: 0.3314 | LR: 1.00e-04
Epoch   4 | Train Loss: 0.2425 | Val Loss: 0.2225 | LR: 1.00e-04
Epoch   5 | Train Loss: 0.1973 | Val Loss: 0.1807 | LR: 1.00e-04
Epoch   6 | Train Loss: 0.1701 | Val Loss: 0.1567 | LR: 1.00e-04
Epoch   7 | Train Loss: 0.1494 | Val Loss: 0.1380 | LR: 1.00e-04
Epoch   8 | Train Loss: 0.1334 | Val Loss: 0.1291 | LR: 1.00e-04
Epoch   9 | Train Loss: 0.1221 | Val Loss: 0.1180 | LR: 1.00e-04
Epoch  10 | Train Loss: 0.1130 | Val Loss: 0.1100 | LR: 1.00e-04
Epoch  11 | Train Loss: 0.1027 | Val Loss: 0.1030 | LR: 1.00e-04
Epoch  12 | Train Loss: 0.0973 | Val Loss: 0.1019 | LR: 1.00e-04
Epoch  13 | Train Loss: 0.0903 | Val Loss: 0.0943 | LR: 1.00e-04
Epoch  14 | Train Loss: 0.0848 | Val Loss: 0.0897 | LR: 1.00e-04
Epoch  15 | Train Loss: 0.0820 | Val Loss: 0.0864 | LR: 1.00e-04
Epoch  16 | Train Loss: 0